## Imports

# Information Theory - Interactive Tutorial

**MSML610: Advanced Machine Learning**  
**Instructor**: Dr. GP Saggese

## Contents
- [1. Introduction to Entropy](#entropy)
- [2. Joint and Conditional Entropy](#joint-conditional)
- [3. Mutual Information](#mutual-info)
- [4. KL Divergence and Cross-Entropy](#kl-cross)
- [5. Advanced Topics](#advanced)

## Description

This notebook provides an interactive exploration of information theory concepts including entropy, mutual information, KL divergence, and their applications in machine learning. Use the interactive widgets to experiment with different probability distributions and visualize how information-theoretic measures change.

### Install packages

In [3]:
if False:
    !sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet jupyterlab-vim)"
    !jupyter labextension enable

### Import modules

In [ ]:
%load_ext autoreload
%autoreload 2

import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ipywidgets import interact, FloatSlider, IntSlider, widgets
from IPython.display import display, Markdown

# Set plotting style.
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
try:
    import msml610_utils as ut
    ut.config_notebook()
except ImportError:
    print("msml610_utils not available, using default configuration")
    
# Initialize logger.
logging.basicConfig(level=logging.INFO)
_LOG = logging.getLogger(__name__)

<a name='entropy'></a>
# 1. Entropy and Uncertainty

## Definition

**Entropy** $H(X)$ of a discrete random variable $X$ is defined as:

$$H(X) = -\sum_x p(x) \log_2 p(x)$$

## Intuition

- Entropy quantifies the average level of **information**, **surprise**, or **uncertainty** inherent in the variable's possible outcomes
- High entropy = more unpredictability
- Low entropy = more certainty
- Unit: bits (when using $\log_2$)

## Examples

- **Fair coin**: Two equally likely outcomes → maximum uncertainty, $H = 1$ bit
- **Biased coin**: If heads occurs 90% of the time → less uncertainty, $H < 1$ bit

In [ ]:
def calculate_entropy(probabilities):
    """
    Calculate Shannon entropy for a discrete probability distribution.
    
    :param probabilities: Array of probabilities (must sum to 1)
    :return: Entropy in bits
    """
    # Filter out zero probabilities to avoid log(0).
    probabilities = np.array(probabilities)
    probabilities = probabilities[probabilities > 0]
    # Calculate entropy using log base 2.
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy


def binary_entropy(p):
    """
    Calculate entropy of a binary random variable.
    
    :param p: Probability of outcome 1
    :return: Binary entropy H(p)
    """
    if p == 0 or p == 1:
        return 0
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


# Test with fair coin.
fair_coin = [0.5, 0.5]
print(f"Fair coin entropy: {calculate_entropy(fair_coin):.4f} bits")

# Test with biased coin.
biased_coin = [0.9, 0.1]
print(f"Biased coin (90-10) entropy: {calculate_entropy(biased_coin):.4f} bits")

# Test with certain outcome.
certain = [1.0, 0.0]
print(f"Certain outcome entropy: {calculate_entropy(certain):.4f} bits")

## Interactive Visualization: Binary Entropy

Use the slider below to adjust the probability $p$ of a binary random variable and observe how entropy changes.

In [ ]:
def plot_binary_entropy_interactive(p=0.5):
    """
    Plot binary entropy function with current value highlighted.
    
    :param p: Probability of outcome 1 (slider controlled)
    """
    # Create array of probabilities.
    p_values = np.linspace(0.001, 0.999, 1000)
    entropy_values = [binary_entropy(p_val) for p_val in p_values]
    # Create figure with two subplots.
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    # Plot 1: Binary entropy function.
    ax1.plot(p_values, entropy_values, 'b-', linewidth=2, label='H(p)')
    ax1.axvline(p, color='red', linestyle='--', linewidth=2, label=f'p = {p:.2f}')
    ax1.axhline(binary_entropy(p), color='red', linestyle='--', linewidth=1, alpha=0.5)
    ax1.scatter([p], [binary_entropy(p)], color='red', s=100, zorder=5)
    ax1.set_xlabel('Probability p', fontsize=12)
    ax1.set_ylabel('Entropy H(p) [bits]', fontsize=12)
    ax1.set_title('Binary Entropy Function', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=11)
    ax1.set_ylim([-0.05, 1.1])
    # Plot 2: Probability distribution.
    outcomes = ['Outcome 0', 'Outcome 1']
    probs = [1-p, p]
    colors = ['skyblue', 'coral']
    ax2.bar(outcomes, probs, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_ylabel('Probability', fontsize=12)
    ax2.set_title(f'Probability Distribution\nEntropy = {binary_entropy(p):.4f} bits', fontsize=14)
    ax2.set_ylim([0, 1.1])
    ax2.grid(True, alpha=0.3, axis='y')
    # Add probability values on bars.
    for i, (outcome, prob) in enumerate(zip(outcomes, probs)):
        ax2.text(i, prob + 0.02, f'{prob:.2f}', ha='center', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()
    # Print information content.
    if p > 0 and p < 1:
        info_0 = -np.log2(1-p)
        info_1 = -np.log2(p)
        print(f"Information content of Outcome 0: {info_0:.4f} bits")
        print(f"Information content of Outcome 1: {info_1:.4f} bits")
        print(f"Expected information (Entropy): {binary_entropy(p):.4f} bits")
    

# Create interactive widget.
interact(plot_binary_entropy_interactive, 
         p=FloatSlider(min=0.01, max=0.99, step=0.01, value=0.5, 
                      description='Probability p:', style={'description_width': 'initial'}));

<a name='joint-conditional'></a>
# 2. Joint and Conditional Entropy

## Joint Entropy

**Joint entropy** $H(X, Y)$ of two variables $X$ and $Y$:

$$H(X, Y) = -\sum_{x,y} p(x,y) \log_2 p(x,y)$$

- Describes the information needed for the joint distribution of $X$ and $Y$
- For independent variables: $H(X, Y) = H(X) + H(Y)$

## Conditional Entropy

**Conditional entropy** $H(Y|X)$ measures uncertainty in $Y$ after observing $X$:

$$H(Y|X) = -\sum_{x,y} p(x,y) \log_2 p(y|x) = \sum_x p(x) H(Y|X=x)$$

**Properties:**
- Low $H(Y|X)$ implies $X$ has strong predictive power for $Y$
- If $Y = X$, then $H(Y|X) = 0$ (no uncertainty)
- If $X$ and $Y$ are independent, then $H(Y|X) = H(Y)$

## Chain Rule for Entropy

$$H(X, Y) = H(X) + H(Y|X) = H(Y) + H(X|Y)$$

In [ ]:
def calculate_joint_entropy(joint_prob):
    """
    Calculate joint entropy H(X,Y) from joint probability distribution.
    
    :param joint_prob: 2D array of joint probabilities p(x,y)
    :return: Joint entropy in bits
    """
    joint_prob = np.array(joint_prob)
    # Filter out zero probabilities.
    joint_prob_flat = joint_prob.flatten()
    joint_prob_flat = joint_prob_flat[joint_prob_flat > 0]
    return -np.sum(joint_prob_flat * np.log2(joint_prob_flat))


def calculate_conditional_entropy(joint_prob):
    """
    Calculate conditional entropy H(Y|X) from joint probability distribution.
    
    :param joint_prob: 2D array of joint probabilities p(x,y)
    :return: Conditional entropy H(Y|X) in bits
    """
    joint_prob = np.array(joint_prob)
    # Calculate marginal p(x).
    p_x = joint_prob.sum(axis=1)
    conditional_entropy = 0
    for i, px in enumerate(p_x):
        if px > 0:
            # Calculate p(y|x) for this x.
            p_y_given_x = joint_prob[i, :] / px
            # Calculate H(Y|X=x).
            p_y_given_x = p_y_given_x[p_y_given_x > 0]
            h_y_given_x = -np.sum(p_y_given_x * np.log2(p_y_given_x))
            # Weight by p(x).
            conditional_entropy += px * h_y_given_x
    return conditional_entropy


# Example: Weather and Activity.
# X = Weather (0: Sunny, 1: Rainy)
# Y = Activity (0: Park, 1: Cinema)
joint_prob = np.array([
    [0.35, 0.15],  # Sunny: 35% park, 15% cinema
    [0.10, 0.40]   # Rainy: 10% park, 40% cinema
])

print("Joint Probability Distribution:")
print(pd.DataFrame(joint_prob, 
                   index=['Sunny', 'Rainy'], 
                   columns=['Park', 'Cinema']))
print()

# Calculate marginals.
p_weather = joint_prob.sum(axis=1)
p_activity = joint_prob.sum(axis=0)

h_weather = calculate_entropy(p_weather)
h_activity = calculate_entropy(p_activity)
h_joint = calculate_joint_entropy(joint_prob)
h_activity_given_weather = calculate_conditional_entropy(joint_prob)

print(f"H(Weather) = {h_weather:.4f} bits")
print(f"H(Activity) = {h_activity:.4f} bits")
print(f"H(Weather, Activity) = {h_joint:.4f} bits")
print(f"H(Activity|Weather) = {h_activity_given_weather:.4f} bits")
print()
print(f"Chain rule verification: H(Weather) + H(Activity|Weather) = {h_weather + h_activity_given_weather:.4f} bits")
print(f"Should equal H(Weather, Activity) = {h_joint:.4f} bits")

<a name='mutual-info'></a>
# 3. Mutual Information

## Definition

**Mutual information** $I(X;Y)$ measures how much knowing one variable reduces uncertainty about the other:

$$I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X) = H(X) + H(Y) - H(X,Y)$$

## Intuition

- Measures the shared information between two variables
- Quantifies the reduction in uncertainty about one variable given the other
- Symmetric: $I(X;Y) = I(Y;X)$

## Properties

- Non-negative: $I(X;Y) \geq 0$
- $I(X;Y) = 0$ if and only if $X$ and $Y$ are independent
- $I(X;X) = H(X)$ (self-information equals entropy)
- Higher mutual information indicates stronger relationship

## Applications

- Feature selection in machine learning
- Dimensionality reduction
- Dependency detection in data

In [ ]:
def calculate_mutual_information(joint_prob):
    """
    Calculate mutual information I(X;Y) from joint probability distribution.
    
    :param joint_prob: 2D array of joint probabilities p(x,y)
    :return: Mutual information in bits
    """
    joint_prob = np.array(joint_prob)
    # Calculate marginals.
    p_x = joint_prob.sum(axis=1)
    p_y = joint_prob.sum(axis=0)
    # Calculate entropies.
    h_x = calculate_entropy(p_x)
    h_y = calculate_entropy(p_y)
    h_xy = calculate_joint_entropy(joint_prob)
    # Mutual information.
    mi = h_x + h_y - h_xy
    return mi


def visualize_information_decomposition(joint_prob):
    """
    Visualize the decomposition of joint entropy into components.
    
    :param joint_prob: 2D array of joint probabilities
    """
    joint_prob = np.array(joint_prob)
    # Calculate all components.
    p_x = joint_prob.sum(axis=1)
    p_y = joint_prob.sum(axis=0)
    h_x = calculate_entropy(p_x)
    h_y = calculate_entropy(p_y)
    h_xy = calculate_joint_entropy(joint_prob)
    h_y_given_x = calculate_conditional_entropy(joint_prob)
    h_x_given_y = calculate_conditional_entropy(joint_prob.T)
    mi = calculate_mutual_information(joint_prob)
    # Create Venn diagram visualization.
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    # Plot 1: Information measures.
    measures = ['H(X)', 'H(Y)', 'H(X,Y)', 'H(Y|X)', 'H(X|Y)', 'I(X;Y)']
    values = [h_x, h_y, h_xy, h_y_given_x, h_x_given_y, mi]
    colors_bars = ['steelblue', 'coral', 'purple', 'lightblue', 'lightsalmon', 'green']
    bars = ax1.bar(measures, values, color=colors_bars, alpha=0.7, edgecolor='black')
    ax1.set_ylabel('Information [bits]', fontsize=12)
    ax1.set_title('Information Measures', fontsize=14)
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.tick_params(axis='x', rotation=45)
    # Add values on bars.
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    # Plot 2: Relationship diagram.
    ax2.text(0.5, 0.9, 'Information Relationships', ha='center', fontsize=16, fontweight='bold')
    ax2.text(0.5, 0.75, f'H(X, Y) = H(X) + H(Y|X)', ha='center', fontsize=12)
    ax2.text(0.5, 0.68, f'{h_xy:.3f} = {h_x:.3f} + {h_y_given_x:.3f}', ha='center', fontsize=11, color='blue')
    ax2.text(0.5, 0.58, f'I(X;Y) = H(X) + H(Y) - H(X,Y)', ha='center', fontsize=12)
    ax2.text(0.5, 0.51, f'{mi:.3f} = {h_x:.3f} + {h_y:.3f} - {h_xy:.3f}', ha='center', fontsize=11, color='green')
    ax2.text(0.5, 0.41, f'I(X;Y) = H(Y) - H(Y|X)', ha='center', fontsize=12)
    ax2.text(0.5, 0.34, f'{mi:.3f} = {h_y:.3f} - {h_y_given_x:.3f}', ha='center', fontsize=11, color='green')
    ax2.text(0.5, 0.24, 'Interpretation:', ha='center', fontsize=12, fontweight='bold')
    ax2.text(0.5, 0.17, f'Knowing X reduces uncertainty about Y by {mi:.3f} bits', ha='center', fontsize=11)
    ax2.text(0.5, 0.10, f'({(mi/h_y)*100:.1f}% of total uncertainty in Y)', ha='center', fontsize=11, style='italic')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()


# Use the weather-activity example.
print("Example: Weather and Activity Correlation")
print("=" * 50)
visualize_information_decomposition(joint_prob)

# Calculate and display mutual information.
mi = calculate_mutual_information(joint_prob)
print(f"\nMutual Information I(Weather; Activity) = {mi:.4f} bits")
print(f"This means knowing the weather reduces uncertainty about activity by {mi:.4f} bits")

## Interactive Visualization: Correlation and Mutual Information

Adjust the correlation strength to see how it affects mutual information between two variables.

In [ ]:
def create_correlated_joint_distribution(correlation=0.5):
    """
    Create a 2x2 joint distribution with specified correlation.
    
    :param correlation: Correlation strength (0=independent, 1=perfectly correlated)
    :return: 2x2 joint probability matrix
    """
    # Create joint distribution with correlation.
    p11 = 0.25 + correlation * 0.25
    p00 = 0.25 + correlation * 0.25
    p10 = 0.25 - correlation * 0.25
    p01 = 0.25 - correlation * 0.25
    joint_prob = np.array([[p00, p01], [p10, p11]])
    return joint_prob


def plot_mutual_info_interactive(correlation=0.5):
    """
    Interactive plot showing how correlation affects mutual information.
    
    :param correlation: Correlation strength between variables
    """
    joint_prob = create_correlated_joint_distribution(correlation)
    # Calculate metrics.
    mi = calculate_mutual_information(joint_prob)
    p_x = joint_prob.sum(axis=1)
    p_y = joint_prob.sum(axis=0)
    h_x = calculate_entropy(p_x)
    h_y = calculate_entropy(p_y)
    # Create visualization.
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))
    # Plot 1: Joint distribution heatmap.
    im = ax1.imshow(joint_prob, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.5)
    ax1.set_xticks([0, 1])
    ax1.set_yticks([0, 1])
    ax1.set_xticklabels(['Y=0', 'Y=1'])
    ax1.set_yticklabels(['X=0', 'X=1'])
    ax1.set_xlabel('Variable Y', fontsize=12)
    ax1.set_ylabel('Variable X', fontsize=12)
    ax1.set_title(f'Joint Distribution p(X,Y)\nCorrelation = {correlation:.2f}', fontsize=14)
    # Add text annotations.
    for i in range(2):
        for j in range(2):
            text = ax1.text(j, i, f'{joint_prob[i, j]:.3f}',
                          ha="center", va="center", color="black", fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax1)
    # Plot 2: Marginal distributions.
    x_pos = np.arange(2)
    width = 0.35
    ax2.bar(x_pos - width/2, p_x, width, label='P(X)', alpha=0.7, color='steelblue')
    ax2.bar(x_pos + width/2, p_y, width, label='P(Y)', alpha=0.7, color='coral')
    ax2.set_xlabel('Outcome', fontsize=12)
    ax2.set_ylabel('Probability', fontsize=12)
    ax2.set_title('Marginal Distributions', fontsize=14)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(['0', '1'])
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis='y')
    # Plot 3: Information metrics vs correlation.
    correlations = np.linspace(0, 1, 50)
    mis = []
    for corr in correlations:
        jp = create_correlated_joint_distribution(corr)
        mis.append(calculate_mutual_information(jp))
    ax3.plot(correlations, mis, 'b-', linewidth=2, label='I(X;Y)')
    ax3.axvline(correlation, color='red', linestyle='--', linewidth=2, label=f'Current: {correlation:.2f}')
    ax3.scatter([correlation], [mi], color='red', s=100, zorder=5)
    ax3.set_xlabel('Correlation', fontsize=12)
    ax3.set_ylabel('Mutual Information [bits]', fontsize=12)
    ax3.set_title('Mutual Information vs Correlation', fontsize=14)
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    ax3.set_ylim([0, 1])
    plt.tight_layout()
    plt.show()
    # Print metrics.
    print(f"Entropy H(X) = {h_x:.4f} bits")
    print(f"Entropy H(Y) = {h_y:.4f} bits")
    print(f"Mutual Information I(X;Y) = {mi:.4f} bits")
    if h_y > 0:
        print(f"Percentage of Y's entropy explained by X: {(mi/h_y)*100:.2f}%")


# Create interactive widget.
interact(plot_mutual_info_interactive,
         correlation=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5,
                                description='Correlation:', style={'description_width': 'initial'}));